# SignalForge Path B ORPO Training (Google Colab)

This notebook trains the SignalForge Path B model with the refreshed preference dataset.

What is different in this version:

- it regenerates the current repo export in Colab
- it uses the improved preference bundle with three hard negatives per task
- it trains on a combined non-held-out tuning pool so structured task types show up during training
- it saves adapters, tokenizer files, and metrics to Google Drive
- it includes a quick smoke-generation pass on the dev split before you touch held-out


In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install --no-cache-dir unsloth


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

REPO_URL = 'https://github.com/nebiyuephrata/SignalForge.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/content/SignalForge')
DRIVE_ROOT = Path('/content/drive/MyDrive/signalforge_orpo_runs')
RUN_NAME = 'path_b_orpo_qwen25_15b'
RUN_DIR = DRIVE_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f'Reusing repo at {REPO_ROOT}')

print({'repo_root': str(REPO_ROOT), 'run_dir': str(RUN_DIR)})

In [ ]:
%cd /content/SignalForge
!python generation_scripts/prepare_preference_data.py
!python training/export_unsloth_datasets.py


In [ ]:
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

TRAIN_PATH = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_train.jsonl'
DEV_PATH = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_dev.jsonl'
HELD_OUT_PATH = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_held_out.jsonl'
MANIFEST_PATH = REPO_ROOT / 'training_data' / 'unsloth' / 'manifest.json'

train_rows = read_jsonl(TRAIN_PATH)
dev_rows = read_jsonl(DEV_PATH)
held_out_rows = read_jsonl(HELD_OUT_PATH)
manifest = json.loads(MANIFEST_PATH.read_text())

print(json.dumps(manifest, indent=2))
pd.DataFrame(train_rows)[['task_type', 'dimension', 'rejection_strategy', 'benchmark_source_split']].value_counts().head(20)

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MODEL_REVISION = '989aa79'
MAX_SEQ_LENGTH = 1536
RUN_SFT_WARMSTART = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    revision=MODEL_REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

train_pref = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
        'rejection_strategy': row['rejection_strategy'],
    }
    for row in train_rows
])

dev_pref = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
        'rejection_strategy': row['rejection_strategy'],
    }
    for row in dev_rows
])

print(train_pref)
print(dev_pref)
print({'bf16': torch.cuda.is_bf16_supported(), 'device': torch.cuda.get_device_name(0)})

In [ ]:
if RUN_SFT_WARMSTART:
    from trl import SFTTrainer, SFTConfig

    warm_train = Dataset.from_list([{'text': row['sft_text']} for row in train_rows])
    warm_dev = Dataset.from_list([{'text': row['sft_text']} for row in dev_rows])

    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=warm_train,
        eval_dataset=warm_dev,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        args=SFTConfig(
            output_dir=str(RUN_DIR / 'sft_warmstart'),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=80,
            learning_rate=2e-4,
            logging_steps=5,
            eval_strategy='steps',
            eval_steps=20,
            save_steps=40,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            report_to='none',
        ),
    )
    sft_result = sft_trainer.train()
    print(sft_result)
else:
    print('Skipping warm-start SFT.')


In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_args = ORPOConfig(
    output_dir=str(RUN_DIR / 'orpo_run'),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=1,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=25,
    save_steps=50,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1024,
    beta=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
)

trainer = ORPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_pref,
    eval_dataset=dev_pref,
    args=orpo_args,
)

train_result = trainer.train()
metrics = dict(train_result.metrics)
metrics['train_rows'] = len(train_rows)
metrics['dev_rows'] = len(dev_rows)
metrics['held_out_rows'] = len(held_out_rows)
metrics['model_name'] = MODEL_NAME
metrics['model_revision'] = MODEL_REVISION

print(metrics)

In [ ]:
adapter_dir = RUN_DIR / 'adapter'
tokenizer_dir = RUN_DIR / 'tokenizer'
adapter_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(tokenizer_dir))
(RUN_DIR / 'train_metrics.json').write_text(json.dumps(metrics, indent=2))
(RUN_DIR / 'manifest_snapshot.json').write_text(json.dumps(manifest, indent=2))

print({'adapter_dir': str(adapter_dir), 'tokenizer_dir': str(tokenizer_dir)})

In [ ]:
FastLanguageModel.for_inference(model)

def preview_generation(row_idx):
    row = dev_rows[row_idx]
    inputs = tokenizer([row['prompt']], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=180, use_cache=True)
    text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    print('TASK:', row['task_id'])
    print('STRATEGY:', row['rejection_strategy'])
    print(text)
    print('-' * 80)

for idx in [0, 1, 2]:
    preview_generation(idx)


## Next Step

Use the dev split to compare prompt stability, groundedness, and formatting. Keep held-out untouched until you are happy with the recipe.